# Experiment 01: Two-dimensional linear transformations

**Goal:** use Manim to see what a matrix does to the entire coordinate plane.

We will animate this matrix:

```text
    |  1  -1 |
A = |        |
    | -1   2 |
```

This notebook deliberately uses **ordinary Markdown and monospaced matrices** rather than embedded LaTeX equations. That way, the explanations display consistently in GitHub and the browser-based VS Code notebook editor in Codespaces.

Run the cells in order. Each `%%manim` cell generates a video directly beneath the code cell.

## 1. Check the environment

Manim is a Python package, but some of its labeling features also call external programs such as `latex` and `dvisvgm`.

The following check detects those programs. If they are missing, the animations will still run with **coordinate-number labels disabled**. The background grid and basis vectors remain visible.

For the complete setup, follow `README.md`.

In [1]:
import shutil
import numpy as np
import manim
from manim import *

print("Manim version:", manim.__version__)
print("LaTeX executable:", shutil.which("latex") or "not installed")
print("dvisvgm executable:", shutil.which("dvisvgm") or "not installed")

# Coordinate-number labels in Manim may require LaTeX.
# Set to False explicitly if the labels give you trouble.
SHOW_COORDINATES = bool(shutil.which("latex") and shutil.which("dvisvgm"))
print("Show coordinate-number labels:", SHOW_COORDINATES)

config.media_width = "85%"

Manim version: 0.22.1
LaTeX executable: /usr/bin/latex
dvisvgm executable: /usr/bin/dvisvgm
Show coordinate-number labels: True


## 2. Define the matrix and a vector

**Matrix multiplication** determines where every point moves:

```text
    | x' |     |  1  -1 |   | x |
    |    |  =  |        | @ |   |
    | y' |     | -1   2 |   | y |

    x' =  x - y
    y' = -x + 2y
```

The **columns** of the matrix tell us where the standard basis vectors go:

| Original vector | Transformed vector | Interpretation |
|:---|:---|:---|
| `e1 = (1, 0)` | `A @ e1 = (1, -1)` | First column of A |
| `e2 = (0, 1)` | `A @ e2 = (-1, 2)` | Second column of A |

Our arbitrary example vector will be `v = (2, 1)`.

In [2]:
# Edit these four numbers, then rerun the calculations and animations.
A = np.array([
    [ 1.0, -1.0],
    [-1.0,  2.0],
])
v = np.array([2.0, 1.0])

e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])

print("A =\n", A)
print("A @ e1 =", A @ e1)
print("A @ e2 =", A @ e2)
print("A @ v  =", A @ v)
print("det(A) =", round(float(np.linalg.det(A)), 8))

assert np.allclose(A @ e1, A[:, 0])
assert np.allclose(A @ e2, A[:, 1])

A =
 [[ 1. -1.]
 [-1.  2.]]
A @ e1 = [ 1. -1.]
A @ e2 = [-1.  2.]
A @ v  = [1. 0.]
det(A) = 1.0


## 3. Animate the grid, basis vectors, and unit square

Watch three things during the animation:

1. The **blue and green basis vectors** move to the columns of `A`.
2. The **yellow unit square** deforms into a parallelogram.
3. The **purple vector** moves from `(2, 1)` to `(1, 0)`.

Since `det(A) = 1`, the unit square and transformed parallelogram have the same area. Lengths and angles, however, need not be preserved.

The animation is *rendered*, not continuously interactive. Change `A` above and rerun the cell to generate a different video.

In [3]:
%%manim -ql -v WARNING --disable_caching TransformPlane

class TransformPlane(LinearTransformationScene):
    def __init__(self, **kwargs):
        super().__init__(
            show_coordinates=SHOW_COORDINATES,
            show_basis_vectors=True,
            leave_ghost_vectors=True,
            **kwargs,
        )

    def construct(self):
        self.add_unit_square(color=YELLOW, opacity=0.30)
        self.add_vector(v, color=PURPLE)
        self.wait(0.5)
        self.apply_matrix(A, run_time=3)
        self.wait(1)

Manim Community v0.22.1

## 4. Return to the original plane with the inverse

For a matrix with a nonzero determinant, there is an **inverse** matrix. Applying the inverse after the original transformation sends every vector back where it started.

```text
A_inverse @ A = Identity

                  | 1  0 |
Identity (2x2) = |      |
                  | 0  1 |
```

When `det(A) = 0`, there is **no inverse**. The notebook detects that case and skips the reverse animation.

In [ ]:
det_A = float(np.linalg.det(A))
if np.isclose(det_A, 0):
    print("This matrix is singular; no inverse exists.")
else:
    A_inverse = np.linalg.inv(A)
    print("A_inverse =\n", A_inverse)
    print("A_inverse @ A =\n", A_inverse @ A)
    assert np.allclose(A_inverse @ A, np.eye(2))

In [ ]:
%%manim -ql -v WARNING --disable_caching TransformAndUndo

class TransformAndUndo(LinearTransformationScene):
    def __init__(self, **kwargs):
        super().__init__(
            show_coordinates=False,  # Keep the second animation lightweight.
            show_basis_vectors=True,
            leave_ghost_vectors=False,
            **kwargs,
        )

    def construct(self):
        self.add_unit_square(color=YELLOW, opacity=0.25)
        self.add_vector(v, color=PURPLE)
        self.wait(0.5)
        self.apply_matrix(A, run_time=2.5)
        self.wait(0.5)
        if not np.isclose(np.linalg.det(A), 0):
            self.apply_inverse(A, run_time=2.5)
        self.wait(1)

## 5. Eigenvalues and eigenvectors (first look)

An eigenvector is a direction that a transformation does not turn away from its own line. Its eigenvalue describes the scale factor along that direction.

```text
A @ u = lambda * u

u       = eigenvector
lambda  = eigenvalue
```

Our matrix is **symmetric**, so it has orthogonal eigenvector directions. The next cell computes and verifies them numerically; a dedicated eigenvector animation can come next.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(A)
for n, (value, direction) in enumerate(zip(eigenvalues, eigenvectors.T), start=1):
    print(f"Eigenpair {n}")
    print(f"  eigenvalue:  {value:.6f}")
    print(f"  eigenvector: {np.round(direction, 6)}")
    assert np.allclose(A @ direction, value * direction)

## 6. Experiments to try

Replace `A` in section 2 with one of these matrices, then rerun the numerical and animation cells.

| Transformation | Python expression | What to observe |
|:---|:---|:---|
| Identity | `np.eye(2)` | Every vector stays where it was |
| 90-degree rotation | `np.array([[0, -1], [1, 0]])` | The plane turns counterclockwise |
| Horizontal shear | `np.array([[1, 1], [0, 1]])` | Horizontal offsets change with height |
| Reflection | `np.array([[-1, 0], [0, 1]])` | The plane flips across the y-axis |
| Singular map | `np.array([[1, 1], [1, 1]])` | The plane collapses onto a line |

**Questions:** When does the determinant become negative? What happens to the area when its magnitude is greater than 1? Why can a singular transformation not be undone?

### If you see `FileNotFoundError: 'latex'`

Install the TeX Live dependencies from `README.md`, or keep `SHOW_COORDINATES = False` in the environment cell. The notebook's Markdown itself uses no LaTeX and should display correctly even without a TeX installation.